# 01 — Scrape Movie Scripts from IMSDb

This notebook downloads movie scripts from the [Internet Movie Script Database (IMSDb)](https://imsdb.com/) and saves them locally as `.txt` files.

**Run this notebook first** in the pipeline.

**Outputs:**
- `scripts/` — one `.txt` file per downloaded script (~918 files)
- `imsdb_links.csv` — CSV mapping each movie title to its IMSDb URL

## 1. Imports and Scraping Function

Define `obtener_indice_imsdb()`, which fetches the full movie list from IMSDb's index page and maps each title to its script URL.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from thefuzz import process, fuzz
import re

def fetch_imsdb_index():
    """Fetches the full movie index from IMSDb with their script URLs."""
    url = "https://imsdb.com/all-scripts.html"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    
    try:
        res = requests.get(url, headers=headers, timeout=20)
        res.encoding = 'utf-8'
        soup = BeautifulSoup(res.text, 'html.parser')
        
        dict_urls = {}
        
        # Get all links
        all_links = soup.find_all('a', href=True)
        print(f"Total links found: {len(all_links)}")
        
        # Extract only movie script links (those with /Movie Scripts/ in href)
        script_count = 0
        for link in all_links:
            href = link.get('href', '')
            title_raw = link.get_text(strip=True)
            
            # Filter only links pointing to movie scripts
            if '/Movie Scripts/' in href and '.html' in href.lower():
                # Clean the title: remove ratings and extra whitespace
                title = re.sub(r'\s*\d+/10\s*', '', title_raw).strip()
                
                # Validations
                if not title or len(title) < 2 or len(title) > 300:
                    continue
                
                # Build URL with formatted title
                # Format is: /scripts/FORMATTED-TITLE.html
                title_url = title.replace(' ', '-').replace("'", '')
                # Remove special characters
                title_url = re.sub(r'[^a-zA-Z0-9\-]', '', title_url)
                # Remove multiple dashes
                title_url = re.sub(r'-+', '-', title_url)
                
                final_url = f"https://imsdb.com/scripts/{title_url}.html"
                
                # Add to dictionary (avoid duplicates)
                if title not in dict_urls:
                    dict_urls[title] = final_url
                    script_count += 1
        
        print(f"✅ Extracted {script_count} movie titles")
        print(f"✅ Retrieved {len(dict_urls)} unique movies from IMSDb\n")
        
        return dict_urls
        
    except Exception as e:
        print(f"❌ Error fetching index: {e}")
        return {}


## 2. Diagnose the IMSDb Index

Fetch the index and inspect the first few results to verify the URL extraction is working.

> **Note:** The first 5 entries are malformed (concatenated titles from the page layout) and are dropped when building the link index.

In [ ]:
# DEBUG: Verify the fetched index
print("="*70)
print("INDEX DIAGNOSTICS")
print("="*70)

# Fetch the index
online_index = fetch_imsdb_index()

print(f"\nTotal movies in index: {len(online_index)}")

if len(online_index) > 0:
    print("\nFirst 10 movies and their URLs:")
    print("-"*70)
    for i, (title, url) in enumerate(list(online_index.items())[:10], 1):
        print(f"{i:2d}. {title[:50]}")
        print(f"    URL: {url}")
    
    print("\n" + "="*70)
    print("DOWNLOAD TEST")
    print("="*70)
    
    # Test with the first movie
    first_title, first_url = list(online_index.items())[0]
    print(f"\nTesting download of: {first_title}")
    print(f"URL: {first_url}\n")
    
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        res = requests.get(first_url, headers=headers, timeout=10)
        res.encoding = 'utf-8'
        res.raise_for_status()
        
        print(f"✅ Status code: {res.status_code}")
        print(f"✅ Content length: {len(res.text)} characters")
        
        soup = BeautifulSoup(res.text, 'html.parser')
        pre = soup.find('pre')
        
        if pre:
            text = pre.get_text()
            size = len(text.strip())
            print(f"✅ Found <pre> with {size} characters")
            print(f"\nFirst 300 characters:\n{'-'*70}")
            print(text[:300])
            print(f"{'-'*70}")
        else:
            print("❌ No <pre> found")
            print("\nMain tags on the page:")
            for tag_name in ['pre', 'div', 'p', 'table', 'body']:
                tags = soup.find_all(tag_name)
                if tags:
                    sizes = [len(tag.get_text()) for tag in tags]
                    print(f"  {tag_name}: {len(tags)} found (sizes: {sizes[:3]}...)")
                
    except requests.exceptions.Timeout:
        print("❌ Timeout")
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP Error: {e}")
    except Exception as e:
        print(f"❌ Error: {e}")
else:
    print("❌ No movies retrieved from index")


## 3. Build and Save the Link Index

Filter out the malformed first 5 entries, then save the clean title→URL mapping to `imsdb_links.csv`.

In [ ]:
import pandas as pd

# Create a DataFrame with titles and links
df_links = pd.DataFrame(list(online_index.items()), columns=['Title', 'URL'])

# Skip the first 5 records because they are malformed due to the page layout
df_links_filtered = df_links.iloc[5:].reset_index(drop=True)

# Display all links
print(f"Total movies (excluding first 5): {len(df_links_filtered)}\n")
print(df_links_filtered.to_string())

# Save to CSV
df_links_filtered.to_csv('imsdb_links.csv', index=False)
print(f"\n✓ Saved 'imsdb_links.csv' with all links")


## 4. Test Download — First 10 Scripts

Download a small sample before running the full batch to confirm the download and parsing logic works end-to-end.

In [ ]:
import os
from pathlib import Path

# Create output folder for test scripts
test_folder = "scripts_test"
Path(test_folder).mkdir(exist_ok=True)

print("="*70)
print("DOWNLOADING FIRST 10 SCRIPTS")
print("="*70)

# Get the first 10 records from the filtered DataFrame
first_10 = df_links_filtered.head(10)

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
downloaded = 0
errors = 0

for idx, row in first_10.iterrows():
    title = row['Title']
    url = row['URL']
    
    print(f"\n{idx+1}. Downloading: {title}")
    print(f"   URL: {url}")
    
    try:
        res = requests.get(url, headers=headers, timeout=10)
        res.encoding = 'utf-8'
        res.raise_for_status()
        
        # Find the script on the page
        soup = BeautifulSoup(res.text, 'html.parser')
        pre = soup.find('pre')
        
        if pre:
            text = pre.get_text()
            
            # Create a valid filename
            filename = title.replace(' ', '_').replace('/', '_').lower()
            file_path = os.path.join(test_folder, f"{filename}.txt")
            
            # Save the script
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(text)
            
            size = len(text)
            print(f"   ✅ Downloaded successfully ({size} characters)")
            downloaded += 1
        else:
            print(f"   ❌ No <pre> found on page")
            errors += 1
            
    except requests.exceptions.Timeout:
        print(f"   ❌ Timeout")
        errors += 1
    except requests.exceptions.HTTPError as e:
        print(f"   ❌ HTTP Error: {e.response.status_code}")
        errors += 1
    except Exception as e:
        print(f"   ❌ Error: {str(e)[:50]}")
        errors += 1

print("\n" + "="*70)
print(f"SUMMARY: {downloaded} downloaded, {errors} errors")
print(f"Folder: {os.path.abspath(test_folder)}")
print("="*70)


## 5. Download All Scripts

Download all ~1,293 scripts to the `scripts/` folder. A 0.2s delay is added between requests to avoid overloading the server. Estimated time: ~10 minutes.

In [ ]:
import os
from pathlib import Path
import time

# Folder where all downloaded scripts will be stored
scripts_folder = "scripts"
Path(scripts_folder).mkdir(exist_ok=True)

print("="*70)
print("DOWNLOADING ALL SCRIPTS")
print("="*70)
print(f"Total movies to download: {len(df_links_filtered)}\n")

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
downloaded = 0
errors = 0
start_time = time.time()

for idx, row in df_links_filtered.iterrows():
    title = row['Title']
    url = row['URL']
    
    # Log progress every 50 movies
    if (idx + 1) % 50 == 0:
        elapsed = time.time() - start_time
        print(f"\nProgress: {idx + 1}/{len(df_links_filtered)} downloaded ({elapsed:.1f}s)")
    
    try:
        res = requests.get(url, headers=headers, timeout=10)
        res.encoding = 'utf-8'
        res.raise_for_status()
        
        # Find the script on the page
        soup = BeautifulSoup(res.text, 'html.parser')
        pre = soup.find('pre')
        
        if pre:
            text = pre.get_text()
            
            # Create a valid filename
            filename = title.replace(' ', '_').replace('/', '_').replace("'", '').lower()
            file_path = os.path.join(scripts_folder, f"{filename}.txt")
            
            # Save the script
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(text)
            
            downloaded += 1
        else:
            errors += 1
            
    except requests.exceptions.Timeout:
        errors += 1
    except requests.exceptions.HTTPError:
        errors += 1
    except Exception as e:
        errors += 1
    
    # Small delay to avoid overloading the server
    time.sleep(0.2)

total_time = time.time() - start_time
print("\n" + "="*70)
print(f"FINAL SUMMARY:")
print(f"✅ Scripts downloaded: {downloaded}")
print(f"❌ Errors: {errors}")
print(f"⏱️  Total time: {total_time:.1f} seconds")
print(f"📁 Folder: {os.path.abspath(scripts_folder)}")
print("="*70)


## 6. Validate Downloaded Files

Check for empty files (scripts where IMSDb returned an empty `<pre>` block) and remove them to keep the `scripts/` folder clean.

In [ ]:
import os

print("="*70)
print("CHECKING FOR EMPTY FILES")
print("="*70)

# List all files in the folder
files = os.listdir(scripts_folder)
total_files = len(files)

empty_files = []
files_with_content = []

for file in files:
    file_path = os.path.join(scripts_folder, file)
    
    if os.path.isfile(file_path):
        size = os.path.getsize(file_path)
        
        if size == 0:
            empty_files.append(file)
        else:
            files_with_content.append((file, size))

print(f"\nTotal files: {total_files}")
print(f"✅ Files with content: {len(files_with_content)}")
print(f"❌ Empty files: {len(empty_files)}")

if empty_files:
    print("\n" + "-"*70)
    print("EMPTY FILES FOUND:")
    print("-"*70)
    for file in empty_files:
        print(f"  • {file}")
    
    # Optionally, delete empty files
    print("\nDeleting empty files...")
    for file in empty_files:
        path = os.path.join(scripts_folder, file)
        os.remove(path)
    print(f"✅ Deleted {len(empty_files)} empty files")
else:
    print("\n✅ No empty files found")

print("\n" + "="*70)
print(f"FINAL STATUS: {len(files_with_content)} valid files in folder")
print("="*70)
